In [1]:
!pip install librosa soundfile


***TASK 3.1***

In [2]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
import hashlib
import os
import zlib

os.makedirs("outputs", exist_ok=True)

In [3]:
# List of audio files I want to inspect for any ultrasonic “carrier” energy
FILES = ['Ex3_sound1.wav', 'Ex3_sound2.wav', 'Ex3_sound3.wav', 'Ex3_sound4.wav']

def stft_mag_db(y, sr, n_fft=4096, hop=1024):
    """
    I compute an STFT magnitude spectrogram so I can measure energy by frequency band.
    - S is the linear magnitude (useful for energy averaging)
    - S_db is the same thing in dB (useful if I want to plot/debug visually)
    - freqs maps each FFT bin to its frequency in Hz
    """
    # STFT -> magnitude only (I don't need phase for this energy check)
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop, window="hann"))

    # Convert to dB relative to the max value (handy for plotting, not required for the score)
    S_db = librosa.amplitude_to_db(S, ref=np.max)

    # Frequency axis for STFT bins (so I know what “18 kHz” actually maps to)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    return S, S_db, freqs

def ultrasonic_score(S, freqs, band=(18000, 24000)):
    """
    I score “how ultrasonic” a file looks by averaging STFT magnitude inside a target band.
    I also grab the most dominant frequency in that band as an approximate carrier peak.

    Returns:
      band_energy (float): average linear magnitude in the ultrasonic band
      peak_freq (float|None): the strongest bin frequency in that band (approx carrier)
    """
    lo, hi = band

    # Pick FFT bins that fall inside the requested ultrasonic band
    idx = np.where((freqs >= lo) & (freqs <= hi))[0]

    # If the sample rate is too low (or the band is invalid), there may be no bins here
    if len(idx) == 0:
        return 0.0, None

    # Average energy in the ultrasonic band (linear magnitude, not dB)
    # This gives me a simple “how much stuff exists above ~18 kHz” number.
    band_energy = float(np.mean(S[idx, :]))

    # Find the dominant peak frequency inside the band:
    # 1) average each selected bin across time
    # 2) pick the bin with the highest mean magnitude
    peak_bin = idx[np.argmax(np.mean(S[idx, :], axis=1))]
    peak_freq = float(freqs[peak_bin])

    return band_energy, peak_freq

scores = []

for f in FILES:
    try:
        # Load audio exactly as-is (sr=None keeps original sample rate)
        # mono=True keeps the analysis consistent across files (one channel)
        y, sr = librosa.load(f, sr=None, mono=True)

        # Build the spectrogram + frequency axis
        S, S_db, freqs = stft_mag_db(y, sr)

        # I cap the upper band edge at Nyquist-1 Hz to avoid “band exceeds sr/2” issues
        energy, peak = ultrasonic_score(S, freqs, band=(18000, min(24000, sr/2 - 1)))

        # Store results so I can later pick the most suspicious file
        scores.append((f, energy, peak, sr))

        # Print a quick per-file summary
        print(f"{f:15s} ultrasonic_energy={energy:.6f}  peak_carrier≈{peak} Hz  sr={sr}")

    except FileNotFoundError:
        # If a file isn’t in the working directory, I just skip it cleanly
        print(f"Missing: {f}")

if scores:
    # “Most suspicious” = highest ultrasonic band energy
    suspicious = max(scores, key=lambda t: t[1])
    print("\nSUSPICIOUS FILE:", suspicious[0], "peak_carrier≈", suspicious[2], "Hz")
else:
    # If nothing loaded, there’s nothing meaningful to compare
    print("No files loaded.")


Ex3_sound1.wav  ultrasonic_energy=0.251424  peak_carrier≈22039.2333984375 Hz  sr=44100
Ex3_sound2.wav  ultrasonic_energy=0.000424  peak_carrier≈20859.375 Hz  sr=48000
Ex3_sound3.wav  ultrasonic_energy=0.000579  peak_carrier≈21984.375 Hz  sr=48000
Ex3_sound4.wav  ultrasonic_energy=0.386449  peak_carrier≈19183.59375 Hz  sr=48000

SUSPICIOUS FILE: Ex3_sound4.wav peak_carrier≈ 19183.59375 Hz


In [4]:
def lowpass_fir(y, sr, cutoff_hz=4000, numtaps=801):
    # I’m building a simple low-pass FIR filter using a windowed-sinc design.
    # - numtaps controls how “sharp” the cutoff is (more taps = cleaner, but slower)
    # - cutoff_hz is the highest frequency I want to keep after demodulation
    t = np.arange(numtaps) - (numtaps - 1) / 2

    # This sinc is the ideal low-pass impulse response (infinite length),
    # and the Hamming window makes it finite and reduces ringing.
    h = np.sinc(2 * cutoff_hz * t / sr) * np.hamming(numtaps)

    # Normalizing makes sure the filter has ~unity DC gain (so volume doesn’t collapse)
    h /= np.sum(h)

    # Convolution applies the filter across the whole signal
    return np.convolve(y, h, mode="same")

def demodulate_am_to_audible(in_wav, out_wav="decoded_secret.wav", carrier_hz=None):
    # Load the audio exactly as recorded (no resampling)
    y, sr = librosa.load(in_wav, sr=None, mono=True)
    y = y.astype(np.float32)

    # If I wasn't given a carrier frequency, I try to estimate it automatically
    if carrier_hz is None:
        S, _, freqs = stft_mag_db(y, sr)

        # I search the ultrasonic band where hidden carriers usually live
        _, carrier_hz = ultrasonic_score(S, freqs, band=(18000, min(24000, sr/2 - 1)))

        # If I still can't find one, there's nothing sensible to demodulate
        if carrier_hz is None:
            raise ValueError("Could not estimate carrier frequency automatically.")

    # Time axis in seconds for each sample (needed for the cosine oscillator)
    t = np.arange(len(y), dtype=np.float32) / sr

    # Coherent mixing:
    # Multiplying by cos(2π f t) shifts energy around the carrier down into audible range.
    # For AM signals, this is a straightforward way to “bring the sidebands down”.
    mixed = y * np.cos(2 * np.pi * carrier_hz * t)

    # After mixing, I low-pass so I keep only the speech/code band and drop ultrasonic leftovers.
    decoded = lowpass_fir(mixed, sr, cutoff_hz=4000, numtaps=801)

    # Normalize to a safe range so the output file is easy to listen to
    decoded /= (np.max(np.abs(decoded)) + 1e-9)

    # Write the decoded audio to disk (same sample rate as the input)
    sf.write(out_wav, decoded, sr)

    # Return details so I can log what happened (useful for the report)
    return out_wav, sr, carrier_hz

# Run on the suspicious file I identified earlier (highest ultrasonic energy)
sus_file = suspicious[0]
carrier = suspicious[2]

# Use the detected carrier so I demodulate around the strongest peak I found
out_wav, sr, used_carrier = demodulate_am_to_audible(sus_file, "decoded_secret.wav", carrier_hz=carrier)

print("Saved:", out_wav, "| sr:", sr, "| carrier used:", used_carrier)


Saved: decoded_secret.wav | sr: 48000 | carrier used: 19183.59375


***TASK 3.2***

In [5]:
def read_wav_int16(path):
    # I read the WAV as true 16-bit PCM so I don’t accidentally convert levels/precision.
    # always_2d=False keeps mono files as 1D arrays, which is easier to work with.
    x, sr = sf.read(path, dtype="int16", always_2d=False)

    # If the file is stereo, I just take the left channel to keep the rest of the pipeline simple.
    if x.ndim == 2:  # stereo -> take left channel
        x = x[:, 0]

    # I cast sr to int so downstream code doesn’t get a numpy type unexpectedly.
    return x, int(sr)

def write_wav_int16(path, x_int16, sr):
    # I force the output to be proper 16-bit PCM (PCM_16) so the file is widely compatible.
    # The astype(np.int16) makes sure the dtype matches what soundfile expects for PCM_16.
    sf.write(path, x_int16.astype(np.int16), sr, subtype="PCM_16")


In [6]:
MAGIC = b"STGO"  # I use a fixed 4-byte “signature” so I can quickly tell if decoding worked.

def bytes_to_bits(b: bytes) -> np.ndarray:
    # I convert raw bytes into a 0/1 bit stream (big-endian per byte) for embedding/extraction.
    # np.unpackbits does the heavy lifting here and gives me a clean uint8 array of bits.
    return np.unpackbits(np.frombuffer(b, dtype=np.uint8)).astype(np.uint8)

def bits_to_bytes(bits: np.ndarray) -> bytes:
    # I pack a 0/1 bit stream back into bytes.
    # Only full bytes are valid, so I trim any leftover bits at the end.
    bits = bits.astype(np.uint8)
    n = (len(bits) // 8) * 8
    return np.packbits(bits[:n]).tobytes()

def make_packet(message: str) -> np.ndarray:
    # I turn the message into a self-contained packet:
    # [ MAGIC (4) | LENGTH (4) | CRC32 (4) | PAYLOAD (N) ]
    # That way, when I decode, I can verify I got the right data (and not random noise).
    payload = message.encode("utf-8")
    length = len(payload)

    # CRC32 is a quick integrity check: if even one bit flips, this usually catches it.
    crc = zlib.crc32(payload) & 0xffffffff

    # Header is always 12 bytes: 4 bytes magic + 4 bytes length + 4 bytes CRC32
    header = MAGIC + length.to_bytes(4, "big") + crc.to_bytes(4, "big")
    packet = header + payload

    # Final output is a bit array because that’s what embedding methods usually need.
    return bytes_to_bits(packet)

def parse_packet(packet_bits: np.ndarray) -> str:
    # I reverse the process:
    # 1) bits -> bytes
    # 2) validate header (magic, length, crc)
    # 3) return the decoded UTF-8 payload
    packet_bytes = bits_to_bytes(packet_bits)

    # If I don’t even have the header, the extraction is definitely broken.
    if len(packet_bytes) < 12:
        raise ValueError("Packet too small (corrupt).")

    # Magic must match exactly, otherwise this is the wrong stream or garbage data.
    if packet_bytes[:4] != MAGIC:
        raise ValueError("MAGIC mismatch (wrong key/corrupt extraction).")

    # Read metadata straight from the header
    length = int.from_bytes(packet_bytes[4:8], "big")
    crc_expected = int.from_bytes(packet_bytes[8:12], "big")

    # Pull out just the payload bytes the header claims should exist
    payload = packet_bytes[12:12+length]
    if len(payload) != length:
        raise ValueError("Payload length mismatch (corrupt).")

    # Recompute CRC and compare — this tells me if the payload survived intact.
    crc_actual = zlib.crc32(payload) & 0xffffffff
    if crc_actual != crc_expected:
        raise ValueError("CRC mismatch (wrong key/corrupt extraction).")

    # If everything checks out, I decode as strict UTF-8 (fail fast if it’s not valid text).
    return payload.decode("utf-8", errors="strict")


In [7]:
def prng_indices(n_samples: int, n_needed: int, key: str) -> np.ndarray:
    # I turn the key string into a repeatable numeric seed using SHA-256.
    # Taking the first 8 hex chars gives me a 32-bit-ish seed that’s stable across runs.
    seed = int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:8], 16)

    # I use NumPy’s modern Generator API so randomness is predictable and controlled by the seed.
    rng = np.random.default_rng(seed)

    # I generate a full random permutation of [0..n_samples-1] and then take just what I need.
    # This gives me unique indices with no repeats (useful for “scatter” style embedding).
    return rng.permutation(n_samples)[:n_needed]


In [8]:
def embed_LSB_SEQ_1BIT(in_wav, out_wav, message: str):
    # I embed a message by flipping the Least Significant Bit (LSB) of each audio sample,
    # starting from the beginning of the file (sequential embedding).
    x, sr = read_wav_int16(in_wav)

    # I wrap the message in a packet (MAGIC + length + CRC + payload), then convert it to bits.
    packet_bits = make_packet(message)

    # Each sample can carry exactly 1 bit here (LSB), so capacity = number of samples.
    if len(packet_bits) > len(x):
        raise ValueError("Message too large for this audio (SEQ 1-bit).")

    # Copy the samples so I don’t mutate the original array.
    stego = x.copy()

    # Clear the LSB of each target sample, then OR in the payload bit (0 or 1).
    # This changes the sample value by at most 1, which is usually inaudible.
    stego[:len(packet_bits)] = (stego[:len(packet_bits)] & ~1) | packet_bits

    # Save the modified audio as 16-bit PCM.
    write_wav_int16(out_wav, stego, sr)
    return out_wav

def extract_LSB_SEQ_1BIT(stego_wav):
    # I extract by reading the LSBs back out in the same order I embedded them.
    x, sr = read_wav_int16(stego_wav)

    # The packet header is always 12 bytes = 96 bits.
    # I read just enough bits to reconstruct that header first.
    header_bits = (x[:96] & 1).astype(np.uint8)
    header_bytes = bits_to_bytes(header_bits)

    # Quick sanity check: if the magic doesn't match, this probably isn't stego audio.
    if header_bytes[:4] != MAGIC:
        raise ValueError("MAGIC mismatch (not embedded/corrupt).")

    # Header tells me exactly how many payload bytes exist, so I can compute total packet bits.
    length = int.from_bytes(header_bytes[4:8], "big")
    total_bits = (12 + length) * 8  # header(12) + payload(length), then bytes -> bits

    # Now I read the full packet bitstream and validate/decode it via parse_packet().
    packet_bits = (x[:total_bits] & 1).astype(np.uint8)
    return parse_packet(packet_bits)


In [9]:
def embed_LSB_PRNG_2BIT(in_wav, out_wav, message: str, key: str):
    # I load the audio as 16-bit PCM so I can safely do bit-level edits.
    x, sr = read_wav_int16(in_wav)

    # I wrap the message into a packet (MAGIC + length + CRC + payload), then convert to bits.
    packet_bits = make_packet(message)

    # Here I’m storing 2 bits per sample (2-LSB embedding),
    # so I treat every 2 bits as one “symbol” that goes into one audio sample.
    n_symbols = (len(packet_bits) + 1) // 2
    if n_symbols > len(x):
        raise ValueError("Message too large for this audio (PRNG 2-bit).")

    # I generate a deterministic-but-random-looking set of sample indices from the key.
    # Same key = same index order = extraction works.
    idx = prng_indices(len(x), n_symbols, key)

    # I copy the signal so the input stays untouched.
    stego = x.copy()

    # I walk the PRNG-picked indices and write 2 bits into each chosen sample.
    for i, si in enumerate(idx):
        # Grab the next two bits from the packet (pad with 0 if we’re at the end).
        b0 = packet_bits[2*i] if 2*i < len(packet_bits) else 0
        b1 = packet_bits[2*i+1] if 2*i+1 < len(packet_bits) else 0

        # Pack the two bits into a tiny 0..3 value (b0 is the “high” bit).
        two = (b0 << 1) | b1

        # I clear the last 2 bits of the sample, then drop my 2-bit value into that space.
        stego[si] = (stego[si] & ~3) | two  # I write into the 2 LSBs

    # Save as proper 16-bit PCM so other tools can read it cleanly.
    write_wav_int16(out_wav, stego, sr)
    return out_wav

def extract_LSB_PRNG_2BIT(stego_wav, key: str):
    # I read the stego audio back in the same int16 format used during embedding.
    x, sr = read_wav_int16(stego_wav)

    # The header is fixed-size:
    # 12 bytes = 96 bits, and since I extract 2 bits per sample, that’s 48 samples (“symbols”).
    header_symbols = 48

    # I must use the exact same PRNG index order (same key) to recover the header correctly.
    idx_header = prng_indices(len(x), header_symbols, key)

    # Pull out 2 LSBs from each selected sample, then unpack them into [bit0, bit1].
    header_bits = []
    for si in idx_header:
        two = int(x[si] & 3)
        header_bits.extend([(two >> 1) & 1, two & 1])

    # Trim to exactly 96 bits (just in case) and convert back into bytes.
    header_bits = np.array(header_bits[:96], dtype=np.uint8)

    header_bytes = bits_to_bytes(header_bits)

    # If MAGIC doesn’t match, either the key is wrong or this file wasn’t embedded with this method.
    if header_bytes[:4] != MAGIC:
        raise ValueError("MAGIC mismatch (wrong key/corrupt extraction).")

    # Header tells me how long the payload is, so I know exactly how many bits to read.
    length = int.from_bytes(header_bytes[4:8], "big")
    total_bits = (12 + length) * 8

    # Convert total bits into “symbols” (2 bits per sample).
    total_symbols = (total_bits + 1) // 2

    # Same idea as embedding: regenerate the exact same index list for the full packet.
    idx_all = prng_indices(len(x), total_symbols, key)

    # Extract all 2-bit chunks from the chosen samples and rebuild the full packet bitstream.
    bits = []
    for si in idx_all:
        two = int(x[si] & 3)
        bits.extend([(two >> 1) & 1, two & 1])

    # Slice to the exact packet length and validate/decode using parse_packet().
    packet_bits = np.array(bits[:total_bits], dtype=np.uint8)
    return parse_packet(packet_bits)


In [10]:
def embed_LSB_GAP_REPEAT(in_wav, out_wav, message: str, gap=5, repeat=3):
    # I embed the message using two tricks to make extraction more robust:
    # 1) GAP: I don’t write into every sample — I skip ahead by `gap` samples each time.
    # 2) REPEAT: I write each bit multiple times (`repeat`) so I can majority-vote on extraction.
    x, sr = read_wav_int16(in_wav)

    # I packetize the message (MAGIC + length + CRC + payload) and convert to bits.
    packet_bits = make_packet(message)

    # Repeat each bit (e.g., 3x) to give redundancy against noise/edits.
    bits_rep = np.repeat(packet_bits, repeat)

    # With a fixed gap, the final written position is:
    # pos = 0, gap, 2*gap, ..., (len(bits_rep)-1)*gap
    # so the number of samples I need to cover that span is:
    needed = 1 + (len(bits_rep) - 1) * gap
    if needed > len(x):
        raise ValueError("Message too large for this audio (GAP+REPEAT).")

    # Copy the samples so I don’t mutate the original.
    stego = x.copy()

    # Walk forward with a stride of `gap`, writing one repeated bit at each step.
    pos = 0
    for b in bits_rep:
        # Clear 1 LSB then drop in the bit (0/1). Each write changes the sample by at most 1.
        stego[pos] = (stego[pos] & ~1) | int(b)
        pos += gap

    # Save stego audio as 16-bit PCM.
    write_wav_int16(out_wav, stego, sr)
    return out_wav

def extract_LSB_GAP_REPEAT(stego_wav, gap=5, repeat=3):
    # I extract by sampling every `gap` samples and doing a majority vote across repeats.
    x, sr = read_wav_int16(stego_wav)

    # First I recover the header so I know how big the payload is.
    # Header = 12 bytes = 96 bits, but each bit was written `repeat` times.
    header_reads = 96 * repeat

    # The written bits are spaced by `gap`, so the span I need to read is:
    header_span = 1 + (header_reads - 1) * gap

    # Grab the raw repeated header bits by striding through the signal.
    raw = (x[:header_span:gap] & 1).astype(np.uint8)

    # Collapse repeats back into a single bit using majority vote (mean -> round).
    header_bits = []
    for i in range(96):
        chunk = raw[i*repeat:(i+1)*repeat]
        header_bits.append(int(np.round(np.mean(chunk))))
    header_bits = np.array(header_bits, dtype=np.uint8)

    # Convert header bits back to bytes and sanity-check the magic.
    header_bytes = bits_to_bytes(header_bits)
    if header_bytes[:4] != MAGIC:
        raise ValueError("MAGIC mismatch (corrupt extraction).")

    # Header tells me payload size, so I can compute the full packet size in bits.
    length = int.from_bytes(header_bytes[4:8], "big")
    total_bits = (12 + length) * 8

    # Now do the same thing for the entire packet (again accounting for repetition).
    total_reads = total_bits * repeat
    total_span = 1 + (total_reads - 1) * gap
    raw2 = (x[:total_span:gap] & 1).astype(np.uint8)

    # Majority vote each repeated bit back into a single 0/1 value.
    bits = []
    for i in range(total_bits):
        chunk = raw2[i*repeat:(i+1)*repeat]
        bits.append(int(np.round(np.mean(chunk))))

    # Convert back into a bit array and validate/decode the packet.
    packet_bits = np.array(bits, dtype=np.uint8)
    return parse_packet(packet_bits)


In [11]:
def embed(in_wav, out_wav, message, algo, key="CM3065-my-key", gap=5, repeat=3):
    # This is my simple “router” function:
    # I pick the embedding method based on the algo string, and pass through the right parameters.
    if algo == "LSB_SEQ_1BIT":
        # Sequential 1-bit LSB embedding (writes from the start of the audio)
        return embed_LSB_SEQ_1BIT(in_wav, out_wav, message)

    if algo == "LSB_PRNG_2BIT":
        # PRNG-scattered 2-bit LSB embedding (key controls the sample positions)
        return embed_LSB_PRNG_2BIT(in_wav, out_wav, message, key)

    if algo == "LSB_GAP_REPEAT":
        # Gap + repetition embedding (stride + redundancy for extra robustness)
        return embed_LSB_GAP_REPEAT(in_wav, out_wav, message, gap=gap, repeat=repeat)

    # If the algo name doesn’t match anything I support, fail fast with a clear error.
    raise ValueError("Unknown algo")

def extract(stego_wav, algo, key="CM3065-my-key", gap=5, repeat=3):
    # Same idea as embed(): this routes extraction to the matching algorithm.
    # Important: the settings here (key/gap/repeat) must match what I used during embedding.
    if algo == "LSB_SEQ_1BIT":
        return extract_LSB_SEQ_1BIT(stego_wav)

    if algo == "LSB_PRNG_2BIT":
        return extract_LSB_PRNG_2BIT(stego_wav, key)

    if algo == "LSB_GAP_REPEAT":
        return extract_LSB_GAP_REPEAT(stego_wav, gap=gap, repeat=repeat)

    raise ValueError("Unknown algo")


In [12]:
INPUT_FILE = "Ex3_sound5.wav"
MESSAGE = "It always seems impossible until it's done"

# I pick which embedding method to test here.
# - LSB_SEQ_1BIT: simplest (1 bit/sample, sequential from the start)
# - LSB_PRNG_2BIT: higher capacity + scattered positions controlled by KEY
# - LSB_GAP_REPEAT: lower capacity but more robust (spaced out + majority voting)
ALGO = "LSB_PRNG_2BIT"   # try also: LSB_SEQ_1BIT, LSB_GAP_REPEAT
KEY  = "CM3065-my-key"

# I save the stego file into an outputs folder, and I include the algo name in the filename
# so it’s obvious which method produced which file.
OUT = f"outputs/ex3_2_stego_{ALGO}.wav"

print("Embedding...")
# This call writes the message into the audio using the selected algorithm/settings.
embed(INPUT_FILE, OUT, MESSAGE, ALGO, key=KEY, gap=5, repeat=3)
print("Embedded ->", OUT)

print("\nExtracting...")
# Extraction must use the same algo + parameters (especially KEY / gap / repeat),
# otherwise the bit order won’t line up and the MAGIC/CRC checks will fail.
decoded = extract(OUT, ALGO, key=KEY, gap=5, repeat=3)
print("Extracted message:\n", decoded)


Embedding...
Embedded -> outputs/ex3_2_stego_LSB_PRNG_2BIT.wav

Extracting...
Extracted message:
 It always seems impossible until it's done
